In [2]:
# importing the packages
import numpy as np
from ortools.linear_solver import pywraplp
import time
from warnings import filterwarnings

filterwarnings("ignore")

# reading the cost_matrix and the pair_matrix
cost_matrix = np.loadtxt("../data/cost_matrix/cost.txt")
pair_matrix = np.genfromtxt("../data/pairings/pair_array.txt", delimiter=',', dtype='int')

# determining the number of flights and tasks
num_pairs = pair_matrix.shape[0]
num_flights = pair_matrix.shape[1]
print(num_pairs, num_flights)

80706 240


In [3]:
# define the restricted master problem
def RMP(index, num_flights, pair_matrix, cost_matrix):
    
    pair_matrix = pair_matrix[index]
    cost_matrix = cost_matrix[index].reshape(-1,1)

    # Initializing the MIP Solver
    solver = pywraplp.Solver.CreateSolver("SAT")
    
    # creating the binary allocation variable
    x = np.array([solver.BoolVar("") for i in range(len(index))]).reshape(-1,1)
    
    # create a matrix that is the product of the decision variable and the pair matrix
    result_matrix = x * pair_matrix
    
    # declaring the constraints
    # Uniqueness of flight legs and all flight legs covered
    for i in range(num_flights):
        solver.Add(solver.Sum(result_matrix[:,i]) >= 1)
    
    # declaring the objective function
    solver.Minimize(np.sum(cost_matrix * x))
    
    # Solve the problem
    status = solver.Solve()

    # find the values of the decision variable
    x_values = [x[i][0].solution_value() for i in range(len(index))]

    # find the indices of the final selected pairs
    optimal_index = [value for value, binary_value in zip(index, x_values) if binary_value == 1]

    return status, solver.Objective().Value(), optimal_index

In [8]:
# function to find the reduced cost matrix
def sub_problem(pair_matrix, cost_matrix, index, num_pairs, num_flights):
        
    pair_matrix_idx = pair_matrix[index]
    cost_matrix_idx = cost_matrix[index].reshape(-1,1)
    
    # Initializing the LP Solver
    solver = pywraplp.Solver.CreateSolver("GLOP")
    
    # creating the binary allocation variable
    x = np.array([solver.NumVar(0, 1, f"x_{i}") for i in range (len(index))]).reshape(-1,1)
    
    # create a matrix that is the product of the decision variable and the pair matrix
    result_matrix = x * pair_matrix_idx
    
    # declaring the constraints
    # Uniqueness of flight legs and all flight legs covered
    for i in range(num_flights):
        solver.Add(solver.Sum(result_matrix[:,i]) >= 1)
        
    # declaring the objective function
    solver.Minimize(np.sum(cost_matrix_idx * x))
    
    # Solve the problem
    status = solver.Solve()
    
    # find the values of the decision variable
    dual_values = np.array([constraint.dual_value() for constraint in solver.constraints()])
    
    # find the reduced cost matrix
    red_cost_mtx = []
    
    for i in range(num_pairs):
        red_cost_mtx.append(cost_matrix[i] - np.sum(pair_matrix[i] * dual_values, axis=0))

    return red_cost_mtx, np.argsort(red_cost_mtx)[:50].tolist()

In [12]:
# coloumn generation

# initialization huerestics
ini_pair, increment = 5000, 1000

while True:
    # create a reduced pair array that contains atleast one feasible solution
    index = np.argsort(cost_matrix)[:ini_pair].tolist()

    if np.all(np.sum(pair_matrix[index], axis=0) >= 1) == True:
        break

    ini_pair += increment

status, obj, index_1 = RMP(index, num_flights, pair_matrix, cost_matrix)


In [21]:

index = index_1.copy()

In [22]:

# setting the timer
start_time = time.time()
timeout_seconds = 300

# the column generation problem
while True:

    # the RMP solves the problem for a small number of pairs
    status, obj, idx = RMP(index, num_flights, pair_matrix, cost_matrix)
    print(index)
    
    # the sub problem finds the reduced cost matrix and the index of the least reduced cost
    red_cost_mtx, least_red_cost_index = sub_problem(pair_matrix, cost_matrix, index, num_pairs, num_flights)
    print(least_red_cost_index)
    
    index = index + least_red_cost_index
    print(len(index), obj)
    if all(x >= 0 for x in red_cost_mtx) == True:
        break

    if time.time() - start_time >= timeout_seconds:
        print("Breaking out of the loop after timeout.")
        break

[70584, 49554, 24438, 13266, 68365, 24672, 69374, 26809, 73569, 47186, 72393, 72319, 72315, 73136, 26526, 72800, 72789, 26358, 54348, 62269, 54292, 19926, 57523, 57373, 66766, 68029, 52637, 65470, 22513, 65962, 77409, 77535, 4282, 4316, 76980, 42109, 41200, 78972, 79509, 43427, 42282, 78262, 1984, 78142, 45530, 75463, 75785, 28719, 28724, 75098, 28407, 76376, 34919, 29243, 29076, 29116, 29033, 29009, 80053, 76153, 19913, 19960, 55596, 34565, 57228, 29043, 29264, 29097, 29356, 9942, 6634, 18355, 35355, 39837, 64060, 61497, 48165, 50983, 49378, 21113, 19955, 23600, 28898, 29093, 29022, 6930, 3840, 14637, 11251, 10380]
[67344, 61752, 30496, 30494, 30006, 22152, 30004, 22180, 61780, 61778, 67346, 34478, 67224, 30003, 61759, 48893, 61756, 22178, 44594, 44596, 22159, 64301, 30493, 45084, 45086, 22156, 55185, 69211, 72272, 55509, 30009, 76549, 34674, 48895, 80538, 55095, 55187, 22177, 48803, 69138, 80549, 80555, 69183, 67264, 34480, 49217, 34802, 67270, 30499, 76537]
140 1668.0
[70584, 49554,

Try using the GLOP to for 2 mins 
recurse using the dual values, but record the index of the column used

In [24]:
len(idx)

76

In [ ]:
# changes have been made
